In [ ]:
import re
from collections import Counter

import nltk
import numpy as np
import pandas as pd
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [44]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("averaged_perceptron_tagger_eng")

[nltk_data] Downloading package punkt to /home/arthurpmrs/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/arthurpmrs/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/arthurpmrs/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/arthurpmrs/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [45]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

In [48]:
stop_words

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [50]:
def preprocess_lyrics(text):
    if pd.isna(text):
        return ""

    text = text.lower()
    text = re.sub(r"\b\w*\d\w*\b", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = word_tokenize(text)
    stems = [
        stemmer.stem(token)
        for token in tokens
        if token not in stop_words
    ]

    return " ".join(stems)

def get_word_count(text: str) -> dict:
    tokens = text.split()
    return Counter(token for token in tokens).most_common()

In [51]:
df = pd.read_csv("data/songs_reduced.csv")
df["preprocessed_lyrics"] = df["lyrics"].apply(preprocess_lyrics)
texts = df["preprocessed_lyrics"].fillna("")

In [52]:
# Selecionar 5 músicas aleatoriamente
selected_indices = df.sample(5, random_state=42).index

selected = df.loc[selected_indices]

selected[["artists", "name", "preprocessed_lyrics"]]

,artists,name,preprocessed_lyrics
10650,"[""The Strumbellas""]",In This Life,know season aint chang everyday look like rain...
2041,"[""Therion""]",Land of Canaan,noah curs canaan live servant teshub anger sto...
8668,"[""Damian Marley""]",Julie,juli one truli one love best juli break heart ...
1114,"[""The Beatles""]",Norwegian Wood (This Bird Has Flown) - Remaste...,girl say show room isnt good norwegian wood as...
13902,"[""Demons & Wizards""]",Final Warning,doom testifi there danc death pass pump wave b...


In [53]:
count_vectorizer = CountVectorizer()
X_count = count_vectorizer.fit_transform(texts)
X_count.shape

(20000, 66420)

In [54]:
results_count = []

for idx in selected_indices:
    similarities = cosine_similarity(
        X_count[idx],
        X_count
    ).flatten()

    similarities[idx] = -1

    most_similar_position = similarities.argmax()
    similarity_score = similarities[most_similar_position]

    results_count.append({
        "selected_index": idx,
        "similar_index": most_similar_position,
        "similarity": similarity_score
    })

results_count = pd.DataFrame(results_count)

results_count

,selected_index,similar_index,similarity
0,10650,7011,0.486212
1,2041,8180,0.305823
2,8668,10392,0.524894
3,1114,17919,1.000000
4,13902,13903,0.275174


In [ ]:
for _, row in results_count.iterrows():
    original = df.loc[row["selected_index"]]
    similar = df.loc[row["similar_index"]]

    print(f"Música escolhida: {original['name']} - {original['artists']}")
    print(f"Mais semelhante:  {similar['name']} - {similar['artists']}")
    print(f"Similaridade:     {row['similarity']:.4f}")
    print(" ")
    print(">> Letra da música escolhida")
    print(original['lyrics'][:150].strip(), '...')
    print("\n\n>> Letra da música semelhante")
    print(similar['lyrics'][:150].strip(), '...')
    print("-" * 60)
    print(" ")

    

Música escolhida: In This Life - ["The Strumbellas"]
Mais semelhante:  Something About You - ["James Carter", "BCS"]
Similaridade:     0.4862
 
>> Letra da música escolhida
I know the season's ain't been changing
And everyday it looks like rain
But I keep hoping for that sun

The streets are filled with demons
Lord that's ...


>> Letra da música semelhante
There's something about you

You keep my heart in the clouds
All the time
Making it hard for me down here
Not to lose my mind

You make me feel someth ...
------------------------------------------------------------
 
Música escolhida: Land of Canaan - ["Therion"]
Mais semelhante:  Awesome God / How Great Is Our God - ["Triumphant Quartet"]
Similaridade:     0.3058
 
>> Letra da música escolhida
Noah cursed Canaan to live as a servant
Teshub was angered and the storm
Was sweeping through the land

Land of Gods of wind and sea
Of purple dawns a ...


>> Letra da música semelhante
Our God is an awesome God
He reigns from Heaven above


In [80]:
for _, row in results_count.iterrows():
    original = df.loc[row["selected_index"]][['preprocessed_lyrics']].item()
    similar = df.loc[row["similar_index"]][['preprocessed_lyrics']].item()

    # print(original)
    # print(similar)

    original = get_word_count(original)
    similar = get_word_count(similar)

    print(original)
    print(similar)
    original_words = {word for word, count in original}
    similar_words = {word for word, count in similar}

    intersect = original_words.intersection(similar_words)
    
    print(len(intersect), intersect)
    print(" ")

[('know', 11), ('there', 10), ('someth', 10), ('life', 10), ('oh', 5), ('theyr', 5), ('danc', 5), ('dark', 5), ('night', 3), ('chang', 2), ('look', 2), ('that', 2), ('na', 2), ('still', 2), ('wait', 2), ('us', 2), ('season', 1), ('aint', 1), ('everyday', 1), ('like', 1), ('rain', 1), ('keep', 1), ('hope', 1), ('sun', 1), ('street', 1), ('fill', 1), ('demon', 1), ('lord', 1), ('never', 1), ('gon', 1), ('wan', 1), ('everyon', 1), ('river', 1), ('get', 1), ('low', 1), ('skyscrap', 1), ('cover', 1), ('town', 1), ('work', 1), ('day', 1), ('done', 1), ('peopl', 1), ('dress', 1), ('black', 1), ('car', 1), ('got', 1), ('nowher', 1), ('left', 1), ('run', 1)]
[('someth', 35), ('there', 9), ('make', 5), ('aboooout', 5), ('feel', 4), ('ahh', 2), ('keep', 1), ('heart', 1), ('cloud', 1), ('time', 1), ('hard', 1), ('lose', 1), ('mind', 1), ('cant', 1), ('quit', 1), ('figur', 1), ('realli', 1), ('sort', 1), ('spell', 1), ('hide', 1), ('well', 1), ('your', 1)]
3 {'there', 'keep', 'someth'}
 
[('el', 9)